# Desafio Semana 1 — Visão Computacional
**Residência em IA | UniSENAI — Prof. Matheus Vanzan**

CNN para classificação de imagens de satélite (**EuroSAT/RGB**), comparando 5 experimentos com e sem filtros de pré-processamento OpenCV.


## Seção 0 — Setup e Download do Dataset

In [ ]:
import os
import cv2
import random
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TF version:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

### 0.1 Download do EuroSAT

O servidor original (madm.dfki.de) retorna 403. Baixamos do mirror mantido pelo HuggingFace/TorchGeo, que é publicamente acessível no Colab.


In [ ]:
# URL mantida pelo TorchGeo no HuggingFace (verificada em maio/2026)
EUROSAT_URL = (
    "https://huggingface.co/datasets/torchgeo/eurosat/resolve/"
    "c877bcd43f099cd0196738f714544e355477f3fd/EuroSAT.zip"
)
EXTRACT_DIR = Path("/content/eurosat_raw")
ZIP_PATH    = Path("/content/EuroSAT.zip")

if not EXTRACT_DIR.exists():
    print("Baixando EuroSAT (~89 MB)...")
    urllib.request.urlretrieve(EUROSAT_URL, ZIP_PATH)
    size_mb = ZIP_PATH.stat().st_size / 1e6
    print(f"Download OK ({size_mb:.1f} MB). Extraindo...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    ZIP_PATH.unlink()
    print("Extração concluída.")
else:
    print("EuroSAT já está extraído em", EXTRACT_DIR)

In [ ]:
# Localizar o diretório que contém as pastas de classe
# O zip pode ter estrutura: EuroSAT_RGB/{classe}/ ou 2750/{classe}/
def find_image_root(base: Path) -> Path:
    """Retorna o diretório cujas subpastas contêm imagens."""
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            sub = Path(root) / d
            if any(f.suffix.lower() in ('.jpg', '.png', '.tif')
                   for f in sub.iterdir() if f.is_file()):
                return Path(root)
    return base

IMAGE_ROOT = find_image_root(EXTRACT_DIR)
CLASS_NAMES = sorted(p.name for p in IMAGE_ROOT.iterdir() if p.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Raiz das imagens : {IMAGE_ROOT}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## Seção 1 — Carregamento do Dataset

Usamos `tf.keras.utils.image_dataset_from_directory` que lê diretamente a estrutura de pastas do EuroSAT, sem depender do servidor de download do TFDS.

O split 70 / 15 / 15 é feito depois, com `take` e `skip` no `tf.data.Dataset`.


In [ ]:
IMG_SIZE   = 64
BATCH_SIZE = 32

# Carregar dataset completo sem batch (batch_size=None) para poder fatiar
full_ds = tf.keras.utils.image_dataset_from_directory(
    IMAGE_ROOT,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    shuffle=True,
    seed=SEED,
)

total = sum(1 for _ in full_ds)
n_train = int(0.70 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

ds_train_raw = full_ds.take(n_train)
ds_val_raw   = full_ds.skip(n_train).take(n_val)
ds_test_raw  = full_ds.skip(n_train + n_val)

print(f"Total de imagens : {total}")
print(f"  -> Treino : {n_train}")
print(f"  -> Val    : {n_val}")
print(f"  -> Teste  : {n_test}")

## Seção 2 — Exploração Inicial

### 2.1 Informações do Dataset

In [ ]:
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"IMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE}")
print(f"Total={total} | Train={n_train} | Val={n_val} | Test={n_test}")

### 2.2 Visualização — 2 amostras por classe

In [ ]:
samples_per_class = {i: [] for i in range(NUM_CLASSES)}

for img_t, label_t in ds_train_raw:
    lbl = int(label_t.numpy())
    if len(samples_per_class[lbl]) < 2:
        # img_t é float32 [0,255] — converter para uint8 para imshow
        samples_per_class[lbl].append(img_t.numpy().astype(np.uint8))
    if all(len(v) == 2 for v in samples_per_class.values()):
        break

plt.figure(figsize=(20, 10))
for cls_idx, imgs in samples_per_class.items():
    for j, img in enumerate(imgs):
        pos = cls_idx * 2 + j + 1
        plt.subplot(NUM_CLASSES, 2, pos)
        plt.imshow(img)
        plt.title(CLASS_NAMES[cls_idx], fontsize=8)
        plt.axis('off')
plt.suptitle('EuroSAT — 2 amostras por classe', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 Distribuição de Classes

In [ ]:
# Conta arquivos por pasta de classe (rápido — sem iterar o dataset)
class_counts = {
    cls: len(list((IMAGE_ROOT / cls).iterdir()))
    for cls in CLASS_NAMES
}

plt.figure(figsize=(12, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.xticks(rotation=30, ha='right')
plt.title('Distribuição de classes — EuroSAT (total)')
plt.ylabel('Número de imagens')
plt.tight_layout()
plt.show()
print(class_counts)

## Seção 3 — Pré-processamento e Pipeline

### Por que esses filtros fazem sentido para imagens de satélite?

- **Grayscale:** remove a cor — força o modelo a usar apenas textura/forma. Bom diagnóstico de quanto a cor importa.
- **Canny (bordas):** destaca contornos entre regiões de uso do solo (margens de rios, perímetros urbanos, divisões de cultivos). Conexão direta com a Aula 2.
- **Sharpen (nitidez):** realça detalhes finos (estradas, telhados, divisões de talhões) — potencialmente útil para Highway, Industrial e AnnualCrop.


In [ ]:
def apply_grayscale(img_np):
    """RGB -> gray (1 canal). Entrada e saída: np.uint8."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    return gray[..., np.newaxis]


def apply_canny(img_np, low=50, high=150):
    """Blur + Canny. Kernel 3x3 pois a imagem é 64x64."""
    gray    = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 1.0)
    edges   = cv2.Canny(blurred, low, high)
    return edges[..., np.newaxis]


def apply_sharpen(img_np):
    """Kernel de nitidez (mesmo da Aula 2). Mantém 3 canais."""
    kernel = np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0],
    ], dtype=np.float32)
    return cv2.filter2D(img_np, -1, kernel)

### 3.2 Pipeline reutilizável

`tf.py_function` apaga o shape estático — chamamos `set_shape` logo após para que o `model.fit` saiba a dimensionalidade.

In [ ]:
def make_filter_map(filter_fn, out_channels):
    @tf.autograph.experimental.do_not_convert
    def preprocess_with_filter(img, label):
        # img vem como float32 [0,255] do image_dataset_from_directory
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.uint8)
        if filter_fn is not None:
            img = tf.py_function(
                func=lambda x: filter_fn(x.numpy()),
                inp=[img],
                Tout=tf.uint8,
            )
        img.set_shape([IMG_SIZE, IMG_SIZE, out_channels])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    return preprocess_with_filter


def build_dataset_with_filter(ds_raw, filter_fn, out_channels,
                              batch_size=BATCH_SIZE, shuffle=False):
    fn = make_filter_map(filter_fn, out_channels)
    ds = ds_raw.map(fn, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_dataset(ds_raw, batch_size=BATCH_SIZE, shuffle=False):
    """Pipeline sem filtro (baseline): resize + normalização."""
    return build_dataset_with_filter(ds_raw, filter_fn=None, out_channels=3,
                                     batch_size=batch_size, shuffle=shuffle)

### 3.3 Visualização dos filtros

In [ ]:
# Pegar uma amostra de treino
sample_img_np = None
for img_t, _ in ds_train_raw.take(1):
    sample_img_np = img_t.numpy().astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].imshow(sample_img_np)
axes[0].set_title(f'Original\n{sample_img_np.shape}')
axes[0].axis('off')

gray = apply_grayscale(sample_img_np)
axes[1].imshow(gray[..., 0], cmap='gray')
axes[1].set_title(f'Grayscale\n{gray.shape}')
axes[1].axis('off')

canny = apply_canny(sample_img_np)
axes[2].imshow(canny[..., 0], cmap='gray')
axes[2].set_title(f'Canny\n{canny.shape}')
axes[2].axis('off')

sharp = apply_sharpen(sample_img_np)
axes[3].imshow(sharp)
axes[3].set_title(f'Sharpen\n{sharp.shape}')
axes[3].axis('off')

gray_cv = cv2.cvtColor(sample_img_np, cv2.COLOR_RGB2GRAY)
sx = cv2.Sobel(gray_cv, cv2.CV_64F, 1, 0, ksize=3)
sy = cv2.Sobel(gray_cv, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = cv2.convertScaleAbs(cv2.magnitude(sx, sy))
axes[4].imshow(sobel_mag, cmap='gray')
axes[4].set_title('Sobel Mag\n(visual only)')
axes[4].axis('off')

plt.suptitle('Efeito dos filtros — EuroSAT sample', fontsize=13)
plt.tight_layout()
plt.show()

## Seção 4 — Definição da CNN

Arquitetura única para todos os experimentos — garante comparação justa.

In [ ]:
def build_cnn(input_shape, num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        # Bloco 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # Bloco 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # Bloco 3
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # Classificador
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='eurosat_cnn')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
def run_experiment(name, ds_train, ds_val, ds_test,
                   input_shape, epochs=5):
    print(f"\n{'='*55}")
    print(f"  Experimento : {name}")
    print(f"  Input shape : {input_shape} | Épocas: {epochs}")
    print(f"{'='*55}")

    model = build_cnn(input_shape)
    model.summary()

    history = model.fit(ds_train, validation_data=ds_val,
                        epochs=epochs, verbose=1)

    test_loss, test_acc = model.evaluate(ds_test, verbose=0)
    print(f"\n-> Acurácia no teste: {100 * test_acc:.2f}%")

    return dict(name=name, history=history,
                test_loss=test_loss, test_acc=test_acc,
                epochs=epochs, model=model)

## Seção 5 — Experimentos

| # | Filtro | Épocas | Input | Pergunta respondida |
|---|---|---|---|---|
| 1 | — | 5 | (64,64,3) | Referência |
| 2 | — | 10 | (64,64,3) | Efeito de épocas |
| 3 | Grayscale | 5 | (64,64,1) | Cor é discriminante? |
| 4 | Canny | 5 | (64,64,1) | Bordas ajudam? |
| 5 | Sharpen | 5 | (64,64,3) | Nitidez ajuda? |


### Experimento 1 — Baseline (5 épocas)

In [ ]:
ds_train_base = build_dataset(ds_train_raw, shuffle=True)
ds_val_base   = build_dataset(ds_val_raw)
ds_test_base  = build_dataset(ds_test_raw)

exp1 = run_experiment(
    name='Exp1 — Baseline (5 épocas)',
    ds_train=ds_train_base, ds_val=ds_val_base, ds_test=ds_test_base,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=5,
)

### Experimento 2 — Baseline com 10 épocas

In [ ]:
exp2 = run_experiment(
    name='Exp2 — Baseline (10 épocas)',
    ds_train=ds_train_base, ds_val=ds_val_base, ds_test=ds_test_base,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=10,
)

### Experimento 3 — Grayscale

In [ ]:
ds_train_gray = build_dataset_with_filter(ds_train_raw, apply_grayscale, 1, shuffle=True)
ds_val_gray   = build_dataset_with_filter(ds_val_raw,   apply_grayscale, 1)
ds_test_gray  = build_dataset_with_filter(ds_test_raw,  apply_grayscale, 1)

exp3 = run_experiment(
    name='Exp3 — Grayscale (5 épocas)',
    ds_train=ds_train_gray, ds_val=ds_val_gray, ds_test=ds_test_gray,
    input_shape=(IMG_SIZE, IMG_SIZE, 1), epochs=5,
)

### Experimento 4 — Canny

In [ ]:
ds_train_canny = build_dataset_with_filter(ds_train_raw, apply_canny, 1, shuffle=True)
ds_val_canny   = build_dataset_with_filter(ds_val_raw,   apply_canny, 1)
ds_test_canny  = build_dataset_with_filter(ds_test_raw,  apply_canny, 1)

exp4 = run_experiment(
    name='Exp4 — Canny (5 épocas)',
    ds_train=ds_train_canny, ds_val=ds_val_canny, ds_test=ds_test_canny,
    input_shape=(IMG_SIZE, IMG_SIZE, 1), epochs=5,
)

### Experimento 5 — Sharpen

In [ ]:
ds_train_sharp = build_dataset_with_filter(ds_train_raw, apply_sharpen, 3, shuffle=True)
ds_val_sharp   = build_dataset_with_filter(ds_val_raw,   apply_sharpen, 3)
ds_test_sharp  = build_dataset_with_filter(ds_test_raw,  apply_sharpen, 3)

exp5 = run_experiment(
    name='Exp5 — Sharpen (5 épocas)',
    ds_train=ds_train_sharp, ds_val=ds_val_sharp, ds_test=ds_test_sharp,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=5,
)

## Seção 6 — Tabela de Resultados (7.1)

In [ ]:
experiments = [exp1, exp2, exp3, exp4, exp5]

print(f"{'Experimento':<35} {'Épocas':>6} {'Input Shape':>14} {'Acurácia Teste':>16}")
print("-" * 75)
for exp in experiments:
    shape_str = str(exp['model'].input_shape[1:])
    print(f"{exp['name']:<35} {exp['epochs']:>6} {shape_str:>14} {100*exp['test_acc']:>15.2f}%")

### Curvas de Treinamento

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, exp in enumerate(experiments):
    ax = axes[i]
    h  = exp['history'].history
    ax.plot(h['accuracy'],     label='Train Acc')
    ax.plot(h['val_accuracy'], label='Val Acc')
    ax.set_title(exp['name'])
    ax.set_xlabel('Época')
    ax.set_ylabel('Acurácia')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

axes[5].set_visible(False)
plt.suptitle('Curvas de Treinamento — Todos os Experimentos', fontsize=14)
plt.tight_layout()
plt.show()

### Gráfico Comparativo de Acurácia

In [ ]:
names  = [exp['name'].replace(' — ', '\n') for exp in experiments]
accs   = [100 * exp['test_acc'] for exp in experiments]
colors = ['steelblue', 'steelblue', 'darkorange', 'green', 'purple']

plt.figure(figsize=(12, 5))
bars = plt.bar(names, accs, color=colors, alpha=0.8, edgecolor='black')
plt.ylabel('Acurácia no Teste (%)')
plt.title('Comparativo de Acurácia — EuroSAT')
plt.ylim([0, 100])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Seção 7 — Análise Comparativa (7.2)

### Resultados Observados

| # | Configuração | Épocas | Input | Acurácia Teste |
|---|---|---|---|---|
| 1 | Baseline | 5 | (64,64,3) | 81.51% |
| 2 | Baseline | 10 | (64,64,3) | 79.68% |
| 3 | Grayscale | 5 | (64,64,1) | 75.04% |
| 4 | Canny | 5 | (64,64,1) | 49.65% |
| **5** | **Sharpen** | **5** | **(64,64,3)** | **85.36%** |

### 7.2.1 — Filtros vs Sem Filtros

Ranking: **Sharpen (85.36%) > Baseline (81.51%) > Grayscale (75.04%) >> Canny (49.65%)**.

- **Sharpen venceu o baseline RGB** (+3.85 pp). Diferente da intuição inicial de "menos pré-processamento é melhor", o kernel de nitidez funcionou como *engenharia de feature útil*: mantém os 3 canais de cor (toda a informação cromática preservada) e ainda realça alta-frequência — bordas de talhões agrícolas, contornos de telhados industriais, divisões de estradas. Como o input continua RGB, a rede tem cor + textura + bordas mais nítidas para aprender. Foi o melhor dos dois mundos.

- **Grayscale (-6.47 pp vs baseline)** confirma que a cor é discriminante em EuroSAT: ao colapsar 3 canais em 1, perdemos a separação `Forest (verde) vs HerbaceousVegetation (verde-amarelado) vs Pasture (amarelado)` que depende fortemente do canal de cor. A textura sozinha ainda carrega bastante sinal (75% de acurácia, longe de random), mas paga preço.

- **Canny foi catastrófico (-31.86 pp vs baseline)**: descarta cor *e* textura, deixando apenas um mapa binário de bordas. Para satélite, isso destrói as classes que se distinguem por preenchimento (`Forest`, `Pasture`, `SeaLake`) — ficam com pouquíssimas bordas internas. 49.65% ainda é ~5× random, mas é uma queda enorme. Confirma que Canny é útil como *feature engineering direcionada* (inspeção industrial, OCR), não como pré-processamento genérico para classificação.

### 7.2.2 — Efeito do Número de Épocas

**Exp1 (5 ep, 81.51%) > Exp2 (10 ep, 79.68%)** — surpreendente: dobrar as épocas *piorou* a acurácia.

Isso é um sinal claro de **overfitting incipiente**. Com a CNN simples (sem dropout, sem regularização, sem augmentation) e ~19k imagens de treino, a rede começa a memorizar peculiaridades do conjunto de treino depois de ~5 épocas. A `train_accuracy` continua subindo, mas a `val_accuracy` estabiliza ou cai — e o test set, sendo independente, reflete essa queda. Mais épocas só fariam sentido com regularização (Dropout/BatchNorm) ou augmentation; sem isso, **5 épocas estavam no ponto certo** para esta arquitetura.

### 7.2.3 — Tamanho da Imagem

EuroSAT é nativamente 64×64 (Sentinel-2 RGB).

- **Reduzir para 32×32:** perderíamos detalhes finos que o Sharpen acabou de mostrar serem úteis (estradas, divisões de talhões). Classes como `Highway` e `Industrial`, que dependem de geometria fina, seriam as mais prejudicadas.
- **Ampliar para 128×128 via interpolação:** não adiciona informação real — só duplica pixels. Aumenta custo computacional sem benefício esperado. Só compensaria para usar backbones pré-treinadas (ex.: EfficientNetB0) que exigem entradas maiores.

### 7.2.4 — O que tentaria para melhorar (ancorado nos resultados)

1. **Combinar Sharpen + data augmentation** (`RandomFlip` horizontal/vertical + `RandomRotation`): imagens aéreas são invariantes a rotação/espelhamento, e o Sharpen já provou que features de alta-frequência ajudam — augmentation deve potencializar.
2. **Dropout (~0.4) + BatchNormalization**: o Exp2 mostrou overfitting começando aos 10 ep; com regularização, ganharíamos margem para treinar mais épocas com ganho real.
3. **Grid search no otimizador/lr** (próxima seção): Adam padrão (lr=1e-3) é razoável, mas vale testar AdamW (decoupled weight decay) e SGD+momentum com lrs diferentes.
4. **Transfer learning com EfficientNetB0/MobileNetV2** pré-treinado: provavelmente o maior salto possível (>92% é viável conforme literatura), mas exige redimensionar a 224×224.

### 7.2.5 — Conclusão sobre Pré-processamento

A leitura mais interessante deste experimento é: **filtros clássicos não são uniformemente ruins nem uniformemente bons — depende do que descartam**.

- Filtros que **preservam todos os canais e adicionam realce** (Sharpen) podem ajudar a CNN, atuando como uma camada de feature engineering "grátis".
- Filtros que **descartam canais de informação** (Grayscale, Canny) só fazem sentido quando o canal descartado é comprovadamente irrelevante para a tarefa.

Em EuroSAT, cor é claramente informativa, então Canny destrói o sinal. Em domínios onde o objeto de interesse é definido por geometria (peças industriais sobre fundo uniforme, dígitos manuscritos), a história poderia ser oposta.

Conexão com a Aula 2: filtros clássicos são ferramentas — não receitas. A escolha precisa ser guiada pelo domínio do problema, exatamente como o professor enfatizou.


## Seção 8 — Fase 2: Grid Search no Vencedor (Sharpen)

A Fase 1 mostrou que **Sharpen** foi o filtro vencedor (85.36%). Agora fixamos esse filtro e variamos hiperparâmetros para tentar extrair o máximo da arquitetura.

- **Filtro fixo:** Sharpen (reaproveita `ds_train_sharp`, `ds_val_sharp`, `ds_test_sharp`)
- **Otimizadores:** Adam, AdamW, SGD (momentum=0.9)
- **Learning rates:** 1e-3, 1e-4
- **Épocas:** 10
- **Total:** 6 combinações


In [ ]:
def build_cnn_custom_opt(input_shape, optimizer, num_classes=NUM_CLASSES):
    """Mesma arquitetura do build_cnn, mas com optimizer customizável."""
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='eurosat_cnn_grid')
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def make_optimizer(name, lr):
    if name == 'Adam':
        return keras.optimizers.Adam(learning_rate=lr)
    if name == 'AdamW':
        return keras.optimizers.AdamW(learning_rate=lr)
    if name == 'SGD':
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f'optimizer desconhecido: {name}')

In [ ]:
OPTIMIZERS  = ['Adam', 'AdamW', 'SGD']
LRS         = [1e-3, 1e-4]
GRID_EPOCHS = 10

grid_results = []
for opt_name in OPTIMIZERS:
    for lr in LRS:
        tag = f"{opt_name} lr={lr:.0e}"
        print(f"\n{'='*55}\n  {tag}\n{'='*55}")
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)

        model = build_cnn_custom_opt(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            optimizer=make_optimizer(opt_name, lr),
        )
        hist = model.fit(ds_train_sharp,
                         validation_data=ds_val_sharp,
                         epochs=GRID_EPOCHS, verbose=2)
        test_loss, test_acc = model.evaluate(ds_test_sharp, verbose=0)

        grid_results.append({
            'optimizer': opt_name,
            'lr':        lr,
            'tag':       tag,
            'val_acc':   max(hist.history['val_accuracy']),
            'test_acc':  test_acc,
            'model':     model,
            'history':   hist,
        })
        print(f"-> test_acc = {100*test_acc:.2f}%")

# Tabela final ordenada
grid_results.sort(key=lambda r: r['test_acc'], reverse=True)
print(f"\n{'='*55}")
print(f"{'Configuração':<22} {'Val Acc':>10} {'Test Acc':>10}")
print('-' * 55)
for r in grid_results:
    print(f"{r['tag']:<22} {100*r['val_acc']:>9.2f}% {100*r['test_acc']:>9.2f}%")

best = grid_results[0]
print(f"\n>>> MELHOR: {best['tag']}  ->  test_acc = {100*best['test_acc']:.2f}%")

In [ ]:
grid_matrix = np.zeros((len(OPTIMIZERS), len(LRS)))
for r in grid_results:
    i = OPTIMIZERS.index(r['optimizer'])
    j = LRS.index(r['lr'])
    grid_matrix[i, j] = 100 * r['test_acc']

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(grid_matrix, cmap='YlGn', aspect='auto')
ax.set_xticks(range(len(LRS)))
ax.set_xticklabels([f'{lr:.0e}' for lr in LRS])
ax.set_yticks(range(len(OPTIMIZERS)))
ax.set_yticklabels(OPTIMIZERS)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Otimizador')
for i in range(len(OPTIMIZERS)):
    for j in range(len(LRS)):
        ax.text(j, i, f'{grid_matrix[i, j]:.1f}%',
                ha='center', va='center', fontweight='bold')
ax.set_title('Grid Search — Acurácia de Teste (Sharpen + 10 épocas)')
plt.colorbar(im, ax=ax, label='Acurácia (%)')
plt.tight_layout()
plt.show()

## Seção 9 — Avaliação Final da Melhor Configuração

Métricas detalhadas no conjunto de teste para a melhor combinação do grid:
- Accuracy global
- Precision / Recall / F1 (macro e weighted, além de por-classe)
- Matriz de confusão


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_recall_fscore_support, accuracy_score)

best_model = best['model']

# Coletar y_true e y_pred do test set
y_true, y_pred = [], []
for imgs, labels in ds_test_sharp:
    probs = best_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(labels.numpy().tolist())
y_true = np.array(y_true)
y_pred = np.array(y_pred)

all_labels = list(range(NUM_CLASSES))

acc = accuracy_score(y_true, y_pred)
pm, rm, f1m, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=all_labels, average='macro', zero_division=0)
pw, rw, f1w, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=all_labels, average='weighted', zero_division=0)

print(f"Melhor configuração: {best['tag']}")
print(f"Accuracy : {100*acc:.2f}%")
print(f"Macro    -> P={100*pm:.2f}%  R={100*rm:.2f}%  F1={100*f1m:.2f}%")
print(f"Weighted -> P={100*pw:.2f}%  R={100*rw:.2f}%  F1={100*f1w:.2f}%")
print()
print(classification_report(y_true, y_pred,
                            labels=all_labels,
                            target_names=CLASS_NAMES,
                            digits=3, zero_division=0))

# Matriz de confusão (forçar shape completo com labels=all_labels)
cm = confusion_matrix(y_true, y_pred, labels=all_labels)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predito')
ax.set_ylabel('Verdadeiro')
ax.set_title(f"Matriz de Confusão — {best['tag']}")

thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]),
                ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > thresh else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()